# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`
This notebook demonstrates how to load, process, and explore the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json) using the `mlcroissant` library.

### Dataset Source
The dataset is described using a Croissant schema, available at the URL below.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Identifier: {metadata.identifier}")
print(f"License: {metadata.license}")
print(f"Keywords: {getattr(metadata, 'keywords', [])}")

## 2. Data Overview
Review available record sets and their structure. All items are referenced by their `@id`.

**Note:** We'll list all record set `@id` values, then explore fields and columns within those record sets.

In [ ]:
# List all record sets available in the dataset

record_sets = dataset.record_sets
print(f"Number of record sets: {len(record_sets)}\n")

for rs in record_sets:
    print(f"Record set @id: {rs['@id']}")
    print(f"  Name: {rs.get('name', '[no name]')}")
    print(f"  Description: {rs.get('description', '[no description]')}")
    # List fields by @id within record set
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print(f"  Fields:")
    for f in fields:
        if isinstance(f, dict):
            field_id = f.get('@id')
            name = f.get('name')
        else:
            field_id = f
            name = None
        print(f"    - {field_id} {'' if name is None else '('+name+')'}")
    
    columns = rs.get('column', [])
    if columns:
        print(f"  Columns:")
        for c in columns:
            if isinstance(c, dict):
                print(f"    - {c.get('@id')} ({c.get('name', '')})")
            else:
                print(f"    - {c}")
    print("")

## 3. Data Extraction
For demonstration, we'll extract all tabular data from each record set into Pandas DataFrames. Use the record set and field `@id`s from the overview.

_Note:_ You may change `selected_record_set_id` below to any desired record set `@id` to focus your analysis.

In [ ]:
# Get all record set @id values
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
print('Record sets found:', record_set_ids)

# Load each record set into a dataframe (by @id)
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records from record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Show columns from the first record set for demonstration
if record_set_ids:
    selected_record_set_id = record_set_ids[0]
    print(f"\nColumns in record set {selected_record_set_id}:")
    print(dataframes[selected_record_set_id].columns.tolist())
    dataframes[selected_record_set_id].head()
else:
    print("No record sets found.")

## 4. Exploratory Data Analysis (EDA)
Let's apply some processing: filter, normalize, and group data. For this demonstration, we select the first available numeric column from the chosen record set and operate on it. Please adjust `numeric_field_id` or `group_field_id` if specific fields are of interest.

In [ ]:
# For demonstration, use the first available numeric column, if present
import numpy as np

df = dataframes[selected_record_set_id]

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if numeric_cols:
    numeric_field_id = numeric_cols[0]
    print(f"Selected numeric field: {numeric_field_id}")
    
    threshold = df[numeric_field_id].mean() if df[numeric_field_id].notnull().any() else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())
    
    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    
    # Try to find a likely group field (first object type)
    object_cols = [col for col in df.columns if df[col].dtype==object]
    group_field = object_cols[0] if object_cols else None
    if group_field is not None and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean of {numeric_field_id} by {group_field}:")
        display(grouped_df.head())
else:
    print("No numeric columns found in this record set to perform numeric EDA.")

## 5. Visualization
We'll visualize the distribution of the selected numeric field, if present, and explore relationships with a potential grouping field. Adjust as needed according to your data.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram for numeric field
if numeric_cols:
    plt.figure(figsize=(6, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If group_field available, make boxplot by group
    if group_field is not None and group_field in df.columns:
        plt.figure(figsize=(8, 5))
        sns.boxplot(x=df[group_field], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
We demonstrated how to use `mlcroissant` to load a Croissant-schema-based dataset, inspect its record structure by `@id`, extract tabular data into Pandas DataFrames, perform numeric EDA, and visualize distributions. 

For deeper analysis, inspect the metadata in detail and tailor processing steps to your domain's semantics. All dataset elements are referenced by their Croissant `@id`, ensuring unambiguous exploration and reproducibility.